In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")
DB_URL = os.getenv("DB_URL")
engine = create_engine(DB_URL)

print("✅ Connected to Supabase!")

✅ Connected to Supabase!


In [2]:
results   = pd.read_sql("SELECT * FROM results", engine)
laps      = pd.read_sql("SELECT * FROM laps", engine)
weather   = pd.read_sql("SELECT * FROM weather", engine)
qualifying = pd.read_sql("SELECT * FROM qualifying", engine)

print(f"Results:    {results.shape}")
print(f"Laps:       {laps.shape}")
print(f"Weather:    {weather.shape}")
print(f"Qualifying: {qualifying.shape}")

Results:    (479, 12)
Laps:       (26689, 8)
Weather:    (24, 7)
Qualifying: (480, 7)


In [3]:
winners = results[results["finish_pos"] == 1][["race_name","circuit","date","driver","team","grid_pos"]]
print(winners.to_string(index=False))

                race_name           circuit       date driver            team  grid_pos
    Australian Grand Prix         Melbourne 2025-03-16    NOR         McLaren       1.0
       Chinese Grand Prix          Shanghai 2025-03-23    PIA         McLaren       1.0
      Japanese Grand Prix            Suzuka 2025-04-06    VER Red Bull Racing       1.0
       Bahrain Grand Prix            Sakhir 2025-04-13    PIA         McLaren       1.0
 Saudi Arabian Grand Prix            Jeddah 2025-04-20    PIA         McLaren       2.0
         Miami Grand Prix     Miami Gardens 2025-05-04    PIA         McLaren       4.0
Emilia Romagna Grand Prix             Imola 2025-05-18    VER Red Bull Racing       2.0
        Monaco Grand Prix            Monaco 2025-05-25    NOR         McLaren       1.0
       Spanish Grand Prix         Barcelona 2025-06-01    PIA         McLaren       1.0
      Canadian Grand Prix          Montréal 2025-06-15    RUS        Mercedes       1.0
      Austrian Grand Prix       

In [4]:
fig = px.scatter(
    results,
    x="grid_pos",
    y="finish_pos",
    color="driver",
    hover_data=["race_name","team"],
    title="Qualifying Position vs Race Finish Position (2025)",
    labels={"grid_pos": "Grid Position", "finish_pos": "Finish Position"}
)
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(autorange="reversed")
fig.show()

In [5]:
points = results.groupby("driver")["points"].sum().reset_index()
points = points.sort_values("points", ascending=False)

fig = px.bar(
    points,
    x="driver",
    y="points",
    title="2025 Championship Standings",
    color="points",
    color_continuous_scale="reds"
)
fig.show()

In [6]:
tyre_usage = laps.groupby(["race_id","tyre_compound"]).size().reset_index(name="laps")

fig = px.bar(
    tyre_usage,
    x="race_id",
    y="laps",
    color="tyre_compound",
    title="Tyre Compound Usage Per Race (2025)",
    barmode="stack"
)
fig.update_xaxes(tickangle=45)
fig.show()

In [7]:
avg_laps = laps[laps["lap_time_secs"].notna()].groupby("driver")["lap_time_secs"].mean().reset_index()
avg_laps = avg_laps.sort_values("lap_time_secs")

fig = px.bar(
    avg_laps,
    x="driver",
    y="lap_time_secs",
    title="Average Lap Time Per Driver (2025)",
    labels={"lap_time_secs": "Avg Lap Time (seconds)"}
)
fig.show()

In [8]:
merged = results.merge(weather, on="race_id")
winners_weather = merged[merged["finish_pos"] == 1]

fig = px.scatter(
    winners_weather,
    x="avg_track_temp",
    y="driver",
    size="points",
    color="rainfall",
    title="Race Winners vs Track Temperature & Rainfall",
    labels={"avg_track_temp": "Avg Track Temp (°C)"}
)
fig.show()

In [10]:
pole = results[results["grid_pos"]==1]
pole_wins = pole[pole["finish_pos"]==1]
print(f"Pole to win conversion: {len(pole_wins)}/{len(pole)} = {len(pole_wins)/len(pole)*100:.1f}%")

Pole to win conversion: 16/24 = 66.7%


In [11]:
# ── EDA Summary & Key Findings ─────────────────────────────────────────────

print("=" * 55)
print("       2025 F1 SEASON — EDA KEY FINDINGS")
print("=" * 55)

# 1. championship
print("\n🏆 Championship Top 5:")
top5 = results.groupby("driver")["points"].sum().sort_values(ascending=False).head()
for driver, pts in top5.items():
    print(f"   {driver}: {pts:.0f} pts")

# 2. most wins
print("\n🥇 Most Race Wins:")
wins = results[results["finish_pos"]==1]["driver"].value_counts().head()
for driver, w in wins.items():
    print(f"   {driver}: {w} wins")

# 3. pole to win
pole = results[results["grid_pos"]==1]
pole_wins = pole[pole["finish_pos"]==1]
print(f"\n🎯 Pole Position Win Rate: {len(pole_wins)}/{len(pole)} = {len(pole_wins)/len(pole)*100:.1f}%")

# 4. dominant team
print("\n🚗 Wins by Team:")
team_wins = results[results["finish_pos"]==1]["team"].value_counts()
for team, w in team_wins.items():
    print(f"   {team}: {w} wins")

# 5. ML features conclusion
print("\n📊 Key ML Features identified from EDA:")
print("   ✅ Grid position       — strongest predictor")
print("   ✅ Team                — McLaren dominated")
print("   ✅ Tyre compound       — strategy matters")
print("   ✅ Track temperature   — affects performance")
print("   ✅ Driver win rate     — historical strength")

print("\n✅ EDA Complete — Ready for ML Model!")
print("=" * 55)

       2025 F1 SEASON — EDA KEY FINDINGS

🏆 Championship Top 5:
   NOR: 394 pts
   VER: 389 pts
   PIA: 381 pts
   RUS: 289 pts
   LEC: 225 pts

🥇 Most Race Wins:
   VER: 8 wins
   NOR: 7 wins
   PIA: 7 wins
   RUS: 2 wins

🎯 Pole Position Win Rate: 16/24 = 66.7%

🚗 Wins by Team:
   McLaren: 14 wins
   Red Bull Racing: 8 wins
   Mercedes: 2 wins

📊 Key ML Features identified from EDA:
   ✅ Grid position       — strongest predictor
   ✅ Team                — McLaren dominated
   ✅ Tyre compound       — strategy matters
   ✅ Track temperature   — affects performance
   ✅ Driver win rate     — historical strength

✅ EDA Complete — Ready for ML Model!


In [1]:
print("Rows:", len(qualifying))
print("Unique race-driver pairs:",
      qualifying.drop_duplicates(
          subset=["race_id", "driver"]
      ).shape[0])

NameError: name 'qualifying' is not defined